# Evaluate checkpoint-200

Load the cumulative GRPO LoRA checkpoint, generate four completions for each validation question, calculate accuracy and pass@1, and save every completion to CSV.

In [1]:
!git clone https://github.com/joshsalako/telelogs.git

Cloning into 'telelogs'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 245 (delta 0), reused 2 (delta 0), pack-reused 242 (from 2)
Receiving objects: 100% (245/245), 198.88 MiB | 22.19 MiB/s, done.
Resolving deltas: 100% (102/102), done.
Updating files: 100% (86/86), done.


In [2]:
import os
import sys

print("Python:", sys.executable)

# Install uv using the notebook's actual Python.
# Use only one -q, not -qqq.
!{sys.executable} -m pip install -q --upgrade uv

# Make every uv command install into the notebook environment.
os.environ["UV_SYSTEM_PYTHON"] = "1"

Python: /usr/bin/python3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 80.8 MB/s eta 0:00:00:00:0100:01


In [3]:
!uv pip install --upgrade \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo.git" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth.git" \
    bitsandbytes \
    "xformers==0.0.32.post2" \
    datasets \
    pandas

!uv pip install --upgrade --no-deps \
    "transformers==5.2.0" \
    "tokenizers>=0.22.0,<=0.23.0" \
    "trl==0.22.2" \
    "torchao>=0.16.0" \
    "huggingface-hub>=1.3.0"

Using Python 3.12.13 environment at: /usr
Resolved 102 packages in 14.99s                                      
Prepared 43 packages in 55.67s                                           
Uninstalled 34 packages in 2.02s
Installed 43 packages in 781ms                              
 - aiohttp==3.14.1
 + aiohttp==3.14.3
 - annotated-types==0.7.0
 + annotated-types==0.8.0
 + bitsandbytes==0.50.0
 - certifi==2026.6.17
 + certifi==2026.7.22
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 - dill==0.3.8
 + dill==0.4.0
 - filelock==3.29.7
 + filelock==3.32.0
 - fsspec==2025.3.0
 + fsspec==2025.9.0
 + hf-transfer==0.1.9
 - hf-xet==1.5.1
 + hf-xet==1.5.2
 - huggingface-hub==1.23.0
 + huggingface-hub==1.24.0
 + msgspec==0.21.1
 - numpy==2.0.2
 + numpy==2.5.1
 - nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cudnn-cu12==9.10.2.21
 - nvidia-nccl-cu12==2.28.9
 + nvidia-nccl-cu12==2.27.3
 - pandas==2.2.2
 + pandas==3.0.5
 - pillow==11.3.0
 + pillow==12.3.0
 - protobuf==5.29.6
 + protobuf==7

In [1]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "trl",
    "unsloth",
    "unsloth_zoo",
    "tokenizers",
    "bitsandbytes",
    "xformers",
    "torchao",
    "huggingface-hub",
]

for package in packages:
    try:
        print(f"{package:25s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:25s}: NOT INSTALLED")

torch                    : 2.8.0
transformers             : 5.2.0
trl                      : 0.22.2
unsloth                  : 2026.7.5
unsloth_zoo              : 2026.7.6
tokenizers               : 0.22.2
bitsandbytes             : 0.50.0
xformers                 : 0.0.32.post2
torchao                  : 0.17.0
huggingface-hub          : 1.24.0


## Configuration and paths

In [1]:
import gc
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 8192
MAX_COMPLETION_LENGTH = 1024 * 2
MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH
LORA_RANK = 16
NUM_SAMPLES = 4
BATCH_SIZE = 2
TEMPERATURE = 0.7
TOP_P = 0.95
CHECKPOINT_DIR = "checkpoint-200"
OUTPUT_CSV = "validation_predictions.csv"
PROGRESS_CSV = "validation_predictions.progress.csv"
RESUME = True  # Set to False to start a fresh prediction run.

# Set to a positive integer for a quick smoke test. Keep None for the official
# all-864-question evaluation.
EVAL_LIMIT = None

SYSTEM_PROMPT = (
    "You are a senior telecom root-cause analysis engineer. Analyze the supplied "
    "drive-test and engineering evidence carefully. Follow the candidate identifiers "
    "defined in the user prompt and finish with exactly one selected identifier "
    "enclosed in \\boxed{}."
    "Do not write anything after the boxed identifier."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "A CUDA GPU runtime is required."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


def locate_path(name):
    candidates = [
        Path(name),
        Path("/content/telelogs") / name,
        Path("/root/telelogs") / name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {name}. Upload it to Colab or place it in the cloned repository."
    )


CHECKPOINT_PATH = locate_path(CHECKPOINT_DIR)
VALIDATION_QUESTIONS_PATH = locate_path("data/validation_questions.csv")
VALIDATION_TARGET_PATH = locate_path("data/validation_target.csv")

ADAPTER_FILE = CHECKPOINT_PATH / "adapter_model.safetensors"
if not ADAPTER_FILE.is_file():
    raise FileNotFoundError(f"Adapter weights not found at {ADAPTER_FILE}")

print("Checkpoint:", CHECKPOINT_PATH)
print("Adapter weights:", ADAPTER_FILE)
print("Validation questions:", VALIDATION_QUESTIONS_PATH)
print("Validation targets:", VALIDATION_TARGET_PATH)

GPU: Tesla T4
BF16 supported: True
Checkpoint: /content/telelogs/checkpoint-200
Adapter weights: /content/telelogs/checkpoint-200/adapter_model.safetensors
Validation questions: /content/telelogs/data/validation_questions.csv
Validation targets: /content/telelogs/data/validation_target.csv


## Load the base model and checkpoint

In [2]:
torch.cuda.empty_cache()
gc.collect()

# A100 Optimization: Enable TF32 for faster matrix multiplications
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Patch Qwen3.5 3D position IDs calculation for text-only inputs
try:
    import transformers.models.qwen3_5.modeling_qwen3_5 as qwen3_5_module

    _orig_compute_3d_position_ids = qwen3_5_module.Qwen3_5Model.compute_3d_position_ids

    def _is_empty(x):
        return x is None or (hasattr(x, "numel") and x.numel() == 0)

    def patched_compute_3d_position_ids(
        self, input_ids=None, image_grid_thw=None, video_grid_thw=None, **kwargs
    ):
        if _is_empty(image_grid_thw) and _is_empty(video_grid_thw):
            if hasattr(self, "rope_deltas"):
                self.rope_deltas = None
            return None
        try:
            return _orig_compute_3d_position_ids(
                self,
                input_ids=input_ids,
                image_grid_thw=image_grid_thw,
                video_grid_thw=video_grid_thw,
                **kwargs,
            )
        except Exception:
            if hasattr(self, "rope_deltas"):
                self.rope_deltas = None
            return None

    qwen3_5_module.Qwen3_5Model.compute_3d_position_ids = (
        patched_compute_3d_position_ids
    )
    print("Patched Qwen3_5Model.compute_3d_position_ids successfully")
except Exception as error:
    print("Warning: could not patch Qwen3_5 compute_3d_position_ids:", error)

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    fast_inference=False,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Base model loaded:", MODEL_NAME)

Patched Qwen3_5Model.compute_3d_position_ids successfully
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:226: UserWarning: torchcodec 0.11.0+cu128 is incompatible with torch 2.8.0+cu128; install a matching build with `pip install 'torchcodec>=0.7,<0.8.0'`.
  disable_torchcodec_if_broken()


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Base model loaded: unsloth/Qwen3.5-4B


In [3]:
text_tokenizer = getattr(
    tokenizer, "tokenizer", getattr(tokenizer, "text_tokenizer", tokenizer)
)

In [4]:
from safetensors.torch import load_file as safe_load

adapter_weights = safe_load(str(ADAPTER_FILE))
model_state = model.state_dict()


def normalize_key(key):
    while True:
        changed = False
        for prefix in ["base_model.", "model.", "language_model."]:
            if key.startswith(prefix):
                key = key[len(prefix) :]
                changed = True
        if not changed:
            break
    return key.replace(".default.", ".")


adapter_norm = {normalize_key(key): value for key, value in adapter_weights.items()}

loaded = 0
for key, parameter in model_state.items():
    if "lora_" not in key:
        continue
    norm_key = normalize_key(key)
    if norm_key in adapter_norm:
        parameter.data.copy_(
            adapter_norm[norm_key].to(parameter.device, parameter.dtype)
        )
        loaded += 1

total_lora = sum(1 for key in model_state if "lora_" in key)
print(f"Loaded {loaded}/{total_lora} LoRA parameters from {CHECKPOINT_PATH}")

if loaded == 0 or loaded != total_lora:
    raise RuntimeError(
        f"Failed to load the complete checkpoint: loaded {loaded}/{total_lora} LoRA parameters."
    )

Loaded 256/256 LoRA parameters from /content/telelogs/checkpoint-200


## Load and validate the held-out data

In [5]:
validation_questions = pd.read_csv(VALIDATION_QUESTIONS_PATH)
validation_targets = pd.read_csv(VALIDATION_TARGET_PATH)

if list(validation_questions.columns) != ["ID", "question"]:
    raise ValueError(
        f"validation_questions.csv must contain ID,question; got {list(validation_questions.columns)}"
    )

if list(validation_targets.columns) != ["ID", "Target"]:
    raise ValueError(
        f"validation_target.csv must contain ID,Target; got {list(validation_targets.columns)}"
    )

if len(validation_questions) != 864:
    raise ValueError(
        f"Expected 864 validation questions; got {len(validation_questions)}"
    )

if len(validation_targets) != 864 * NUM_SAMPLES:
    raise ValueError(
        f"Expected {864 * NUM_SAMPLES} validation targets; got {len(validation_targets)}"
    )

if validation_questions[["ID", "question"]].isna().any().any():
    raise ValueError("Validation questions contain missing IDs or questions")

if validation_targets[["ID", "Target"]].isna().any().any():
    raise ValueError("Validation targets contain missing IDs or targets")

if validation_questions["ID"].duplicated().any():
    raise ValueError("Validation questions contain duplicate IDs")

if validation_targets["ID"].duplicated().any():
    raise ValueError("Validation targets contain duplicate IDs")

validation_targets["question_ID"] = validation_targets["ID"].str.replace(
    r"_([1-4])$", "", regex=True
)
validation_targets["sample"] = validation_targets["ID"].str.extract(
    r"_([1-4])$", expand=False
)

if validation_targets["sample"].isna().any():
    raise ValueError("Every validation target ID must end in _1, _2, _3, or _4")

validation_targets["sample"] = validation_targets["sample"].astype(int)

valid_labels = {f"C{i}" for i in range(1, 9)}
if not set(validation_targets["Target"]).issubset(valid_labels):
    raise ValueError("Validation targets must contain only C1-C8 labels")

question_ids = set(validation_questions["ID"])
target_question_ids = set(validation_targets["question_ID"])
if question_ids != target_question_ids:
    raise ValueError("Validation question and target base IDs do not match")

target_groups = validation_targets.groupby("question_ID", sort=False)
for question_id, group in target_groups:
    if set(group["sample"]) != {1, 2, 3, 4}:
        raise ValueError(
            f"{question_id} does not contain target suffixes _1 through _4"
        )
    if group["Target"].nunique() != 1:
        raise ValueError(f"{question_id} contains inconsistent validation targets")

target_lookup = {
    (row.question_ID, row.sample): (row.ID, row.Target)
    for row in validation_targets.itertuples(index=False)
}

validation_records = []
for row in validation_questions.itertuples(index=False):
    target = target_lookup[(row.ID, 1)][1]
    validation_records.append(
        {
            "id": row.ID,
            "question": row.question,
            "target": target,
        }
    )

label_counts = (
    pd.Series([record["target"] for record in validation_records])
    .value_counts()
    .sort_index()
)

expected_counts = pd.Series({f"C{i}": 108 for i in range(1, 9)})
if not label_counts.equals(expected_counts):
    raise ValueError(f"Unexpected validation label balance:\n{label_counts}")

print(
    f"Validated {len(validation_records)} questions and {len(validation_targets)} targets"
)
print("Validation balance:", label_counts.to_dict())

Validated 864 questions and 3456 targets
Validation balance: {'C1': 108, 'C2': 108, 'C3': 108, 'C4': 108, 'C5': 108, 'C6': 108, 'C7': 108, 'C8': 108}


In [6]:
def prompt_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]


prompt_lengths = []

for index, record in enumerate(validation_records):
    try:
        token_ids = text_tokenizer.apply_chat_template(
            prompt_messages(record["question"]),
            add_generation_prompt=True,
            tokenize=True,
            enable_thinking=True,
        )
    except Exception as error:
        raise RuntimeError(f"Failed to tokenize validation record {index}") from error

    prompt_length = len(token_ids)
    if prompt_length > MAX_PROMPT_LENGTH:
        raise ValueError(
            f"Validation record {record['id']} is {prompt_length} prompt tokens; "
            f"maximum is {MAX_PROMPT_LENGTH}."
        )
    prompt_lengths.append(prompt_length)

prompt_lengths_array = np.asarray(prompt_lengths)

print("Prompt-length distribution:")
for percentile in [50, 75, 90, 95, 99, 100]:
    length = np.percentile(prompt_lengths_array, percentile)
    print(f"  {percentile:>3}th percentile: {length:.0f} tokens")

print(f"Longest validation prompt: {max(prompt_lengths)} tokens")

Prompt-length distribution:
   50th percentile: 2442 tokens
   75th percentile: 2562 tokens
   90th percentile: 2705 tokens
   95th percentile: 2767 tokens
   99th percentile: 2897 tokens
  100th percentile: 2943 tokens
Longest validation prompt: 2943 tokens


## Generate four completions per question

In [ ]:
import os

from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)
model.eval()

# Decoder-only batched generation requires left padding.
text_tokenizer.padding_side = "left"
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

# Keep the wrapper and model generation settings synchronized.
if tokenizer is not text_tokenizer:
    tokenizer.padding_side = text_tokenizer.padding_side
    tokenizer.pad_token_id = text_tokenizer.pad_token_id
    tokenizer.eos_token_id = text_tokenizer.eos_token_id

model.generation_config.pad_token_id = text_tokenizer.pad_token_id
model.generation_config.eos_token_id = text_tokenizer.eos_token_id

# Match a terminal boxed C1-C8 label with optional whitespace.
FINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(C[1-8])\s*\}\s*$")

evaluation_records = (
    validation_records if EVAL_LIMIT is None else validation_records[:EVAL_LIMIT]
)

PREDICTION_COLUMNS = [
    "ID",
    "question_ID",
    "sample",
    "target",
    "prediction",
    "correct",
    "format_valid",
    "completion",
]

prediction_order = {}
prediction_metadata = {}
for record in validation_records:
    for sample in range(1, NUM_SAMPLES + 1):
        target_id, target = target_lookup[(record["id"], sample)]
        prediction_order[target_id] = len(prediction_order)
        prediction_metadata[target_id] = (record["id"], sample, target)


def make_prediction(target_id, completion):
    question_id, sample, target = prediction_metadata[target_id]
    completion = str(completion).strip()
    match = FINAL_BOX_RE.search(completion)
    prediction = match.group(1) if match else None
    return {
        "ID": target_id,
        "question_ID": question_id,
        "sample": sample,
        "target": target,
        "prediction": prediction,
        "correct": prediction == target,
        "format_valid": prediction is not None,
        "completion": completion,
    }


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary_path = path.with_name(f"{path.name}.tmp")
    frame.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def save_prediction_progress(prediction_by_id):
    rows = sorted(
        prediction_by_id.values(), key=lambda row: prediction_order[row["ID"]]
    )
    frame = pd.DataFrame(rows, columns=PREDICTION_COLUMNS)
    atomic_write_csv(frame, PROGRESS_CSV)


prediction_by_id = {}
progress_path = Path(PROGRESS_CSV)
if RESUME and progress_path.is_file():
    progress_frame = pd.read_csv(progress_path, keep_default_na=False)
    if list(progress_frame.columns) != PREDICTION_COLUMNS:
        raise ValueError(
            f"{PROGRESS_CSV} has unexpected columns: {list(progress_frame.columns)}"
        )
    if progress_frame["ID"].duplicated().any():
        raise ValueError(f"{PROGRESS_CSV} contains duplicate IDs")

    unknown_ids = set(progress_frame["ID"]) - set(prediction_metadata)
    if unknown_ids:
        raise ValueError(
            f"{PROGRESS_CSV} contains {len(unknown_ids)} unknown validation IDs"
        )

    loaded_by_question = {}
    for row in progress_frame.itertuples(index=False):
        question_id, sample, target = prediction_metadata[row.ID]
        if (
            row.question_ID != question_id
            or int(row.sample) != sample
            or row.target != target
        ):
            raise ValueError(f"{PROGRESS_CSV} metadata does not match {row.ID}")
        loaded_by_question.setdefault(question_id, {})[sample] = make_prediction(
            row.ID, row.completion
        )

    incomplete_questions = []
    for question_id, samples in loaded_by_question.items():
        if set(samples) == set(range(1, NUM_SAMPLES + 1)):
            for prediction in samples.values():
                prediction_by_id[prediction["ID"]] = prediction
        else:
            incomplete_questions.append(question_id)

    if incomplete_questions:
        print(
            f"Regenerating {len(incomplete_questions)} incomplete questions from "
            f"{PROGRESS_CSV}."
        )
    print(
        f"Loaded {len(prediction_by_id) // NUM_SAMPLES} completed questions from "
        f"{PROGRESS_CSV}."
    )
    save_prediction_progress(prediction_by_id)
elif not RESUME:
    if progress_path.is_file():
        print(f"RESUME is disabled; replacing existing {PROGRESS_CSV}.")
    save_prediction_progress(prediction_by_id)

completed_question_ids = {row["question_ID"] for row in prediction_by_id.values()}
remaining_records = [
    record
    for record in evaluation_records
    if record["id"] not in completed_question_ids
]
print(
    f"Validation resume status: {len(evaluation_records) - len(remaining_records)} "
    f"completed, {len(remaining_records)} remaining."
)

progress_bar = tqdm(
    range(0, len(remaining_records), BATCH_SIZE), desc="Validation batches"
)
for batch_start in progress_bar:
    batch = remaining_records[batch_start : batch_start + BATCH_SIZE]
    prompts = [
        text_tokenizer.apply_chat_template(
            prompt_messages(record["question"]),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        for record in batch
    ]

    inputs = text_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
        add_special_tokens=False,
    ).to(model.device)

    # Every generated row includes the full left-padded input width.
    prompt_width = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            num_return_sequences=NUM_SAMPLES,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=text_tokenizer.pad_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )

    completions = text_tokenizer.batch_decode(
        output_ids[:, prompt_width:],
        skip_special_tokens=True,
    )

    expected_completions = len(batch) * NUM_SAMPLES
    if len(completions) != expected_completions:
        raise RuntimeError(
            f"Expected {expected_completions} batch completions; got {len(completions)}"
        )

    for batch_index, record in enumerate(batch):
        completion_start = batch_index * NUM_SAMPLES
        record_completions = completions[
            completion_start : completion_start + NUM_SAMPLES
        ]

        for sample, completion in enumerate(record_completions, 1):
            target_id, target = target_lookup[(record["id"], sample)]
            prediction_by_id[target_id] = make_prediction(target_id, completion)

    save_prediction_progress(prediction_by_id)
    completed_in_scope = len(evaluation_records) - (
        len(remaining_records) - min(batch_start + BATCH_SIZE, len(remaining_records))
    )
    progress_bar.set_postfix(
        saved=completed_in_scope,
        remaining=len(evaluation_records) - completed_in_scope,
    )

selected_target_ids = [
    target_lookup[(record["id"], sample)][0]
    for record in evaluation_records
    for sample in range(1, NUM_SAMPLES + 1)
]
missing_target_ids = [
    target_id for target_id in selected_target_ids if target_id not in prediction_by_id
]
if missing_target_ids:
    raise RuntimeError(
        f"Missing {len(missing_target_ids)} validation completions after inference"
    )

predictions = [prediction_by_id[target_id] for target_id in selected_target_ids]
prediction_frame = pd.DataFrame(predictions, columns=PREDICTION_COLUMNS)
atomic_write_csv(prediction_frame, OUTPUT_CSV)

print(f"Saved {len(prediction_frame)} completions to {OUTPUT_CSV}")

Validation batches:   0%|          | 0/432 [00:00<?, ?it/s]

## Accuracy and pass@1

In [ ]:
expected_rows = len(evaluation_records) * NUM_SAMPLES
if len(prediction_frame) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} prediction rows; got {len(prediction_frame)}"
    )

total = len(prediction_frame)
correct = int(prediction_frame["correct"].sum())
accuracy = correct / total if total else 0.0

# With four independent samples per question, pass@1 is the mean correctness
# across all 864 * 4 samples. It is numerically equal to exact sample accuracy.
pass_at_1 = prediction_frame.groupby("question_ID")["correct"].mean().mean()

format_valid = int(prediction_frame["format_valid"].sum())
format_valid_rate = format_valid / total if total else 0.0

print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
print(f"pass@1:  {pass_at_1:.2%} ({correct}/{total})")
print(f"Valid output format: {format_valid_rate:.2%} ({format_valid}/{total})")
print("Predictions CSV:", OUTPUT_CSV)

per_class = (
    prediction_frame.groupby("target", sort=True)["correct"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "correct", "mean": "accuracy"})
)

display(per_class)
display(prediction_frame.head())

In [ ]:
try:
    from google.colab import files

    files.download(OUTPUT_CSV)
except ImportError:
    print("Not running in Colab; CSV remains in the current directory.")